# SC-IRT quickstart

Scene-Conditioned IRT: per-scene difficulty for end-to-end driving, calibrated
from a 16-planner panel on Bench2Drive.

```bash
git clone https://github.com/jeongtaek1m/SC-IRT.git && cd SC-IRT && pip install -e .
```

## 1. Score the released encoder

Every prediction is out-of-fold: the model never saw the scenario type it is scored on.

`scirt.evaluate` takes any `{route_id: score}` mapping — higher means harder —
so the same three metrics score a difficulty model you built yourself. Scores
must be out-of-fold with respect to the 44 scenario types if you intend to
compare them with the released numbers.

In [ ]:
import scirt

bt = scirt.encoder_predictions()      # {route_id: difficulty}
scirt.evaluate(bt)

## 2. The difficulty anchor and its ceiling

The gold anchor is itself estimated from 16 raters, so no predictor can exceed `sqrt(reliability)` — read every rho against this ceiling.

In [ ]:
import matplotlib.pyplot as plt

gold = scirt.gold()
plt.hist(list(gold.values()), bins=30)
plt.xlabel("difficulty  $\\hat{b}$  (logit)"); plt.ylabel("routes")
plt.title("Bench2Drive: 219-route difficulty spectrum"); plt.show()

scirt.noise_ceiling()   # ~2 min: 20 split-half calibrations

## 3. Evaluate a *new planner* in ~25 rollouts

The tinyBenchmarks use case, for driving: adaptively pick informative routes,
observe pass/fail, and reconstruct the full-bank success rate by p-IRT
(observed outcomes kept, IRT probabilities fill in the rest).

Here we simulate a "new" planner by replaying one panel row.

In [ ]:
from scirt import data

panel = data.read_response_panel()
j = 0                                              # pretend planner 0 is new
truth = {r: panel.y[(r, j)] for r in panel.route_ids if panel.observed(r, j)}

responses = {}
for _ in range(25):
    r = scirt.next_route(responses)                # max 2PL Fisher information
    responses[r] = truth.get(r, 0)                 # one closed-loop rollout

est = scirt.estimate_planner(responses)
true_sr = sum(truth.values()) / len(truth)
print(f"theta = {est['theta']:+.2f} ± {est['se']:.2f}")
print(f"estimated SR = {est['sr_hat']:.3f}   (true SR = {true_sr:.3f})")